# nb_test_cleansed_to_silver_data_mappings
## Test: All tables and records migrated from Cleansed to Silver



In [25]:
%run "/05 - shared-libraries/nb_shared_migrations"

In [26]:
storage_account_name = "deintdlsaueaasqebla"

In [27]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import length, col, lit
from datetime import date, datetime

# Get spark session and delta lake paths
spark = SparkSession.builder.getOrCreate()

In [28]:
import os
_san = storage_account_name
_proto = "abfss"
_suffix = "dfs.core.windows.net"
if not _san:
    raise ValueError("storage_account_name parameter is required but was not provided.")

def _base(c): return f"{_proto}://{c}@{_san}.{_suffix}"

data_lake_paths = get_datalake_paths(storage_account_name)

METADATA_BASE_PATH = data_lake_paths["metadata"]
METADATA_CONTROL_PATH = f"{METADATA_BASE_PATH}/control"
METADATA_CONFIG_PATH = f"{METADATA_BASE_PATH}/config"

SILVER_BASE_PATH = data_lake_paths["silver"]
STEP_SILVER_MASTER_UNIFY = "silver_master_unify" 
STEP_SILVER_TRANSACTIONS_UNIFY = 'silver_transactions_unify'

In [29]:
def _compare_mapped_columns(
    source_table,
    target_table,
    target_pk_col,
    source_pk_col,
    source_column,
    target_column
):    

    mismatches = (
        source_table.alias("s")
        .join(
            target_table.alias("t"),
            source_table[source_pk_col] == target_table[target_pk_col]
        )
        .filter(
            source_table[source_column] != target_table[target_column]         
        )
    )
    # mismatches.show(truncate=False)
    return mismatches

def _validate_column_mappings(df_source, df_target, json_file_path):
    
    config = json.loads(
        mssparkutils.fs.head(json_file_path, 1000000)
    )

    primary_key = config["primary_key"][0]

    source_pk_column = next(
        mapping["source_column"]
        for mapping in config["column_mappings"]
        if mapping["target_column"] == primary_key
    )
    

    for mapping in config["column_mappings"]:

        source_column = mapping["source_column"]
        target_column = mapping["target_column"]

        mismatches = _compare_mapped_columns(
            df_source,
            df_target,
            primary_key,
            source_pk_column,
            source_column,
            target_column
        )
        assert mismatches.count() == 0, (
            f"Mismatch found in "
            f"{source_column} -> {target_column}"
        )

        print(f"✓ {source_column} -> {target_column}")

    print("All column mappings validated successfully.")    

## Get list of tables from transform mappings


In [30]:
transform_path = f"{METADATA_CONTROL_PATH}/transform_businessready_entity_mapping"
df_transform = (
    spark.read.format("delta")
    .load(transform_path)
    .filter(
        (
            (col("step_name") == STEP_SILVER_MASTER_UNIFY) |
            (col("step_name") == STEP_SILVER_TRANSACTIONS_UNIFY)
        ) &
        (col("is_active") == True)
    )
)

## loop through cleansed and unified tables

In [31]:
for row in df_transform.collect():
    source_name = row["source_entity_name"]
    target_name = row["target_entity_name"]
    json_config = row["json_config"]
    step_name = row["step_name"]

    cleansed_table_path = f"{SILVER_BASE_PATH}/cleansed/arms/{source_name}"
    assert mssparkutils.fs.exists(cleansed_table_path), \
        f"Cleansed table does not exist: {cleansed_table_path}"

    if step_name == STEP_SILVER_MASTER_UNIFY:
        unified_table_path = f"{SILVER_BASE_PATH}/unified/master/{target_name}"
        unified_json_file_path = f'{METADATA_CONFIG_PATH}/silver/unified/master/{json_config}'
    else:
        unified_table_path = f"{SILVER_BASE_PATH}/unified/transaction/{target_name}"       
        unified_json_file_path = f'{METADATA_CONFIG_PATH}/silver/unified/transaction/{json_config}' 

    assert mssparkutils.fs.exists(unified_table_path), \
        f"Unified table does not exist: {unified_table_path}"
    assert mssparkutils.fs.exists(unified_table_path), \
        f"Unified json file does not exist: {unified_json_file_path}"

    df_source = spark.read.format("delta").load(cleansed_table_path)

    max_cleansed_at = (
        df_source
        .agg(F.max("mdw_cleansed_at").alias("max_cleansed_at"))
        .first()["max_cleansed_at"]
    )

    batch_run_id = (
        df_source
        .filter(F.col("mdw_cleansed_at") == max_cleansed_at)
        .select("mdw_batch_run_id")
        .first()["mdw_batch_run_id"]
    )

    df_source = df_source.filter(
        col("mdw_batch_run_id") == batch_run_id 
    )

    df_target = spark.read.format("delta").load(unified_table_path).filter(
        (col("mdw_is_current") == True) & 
        (col("mdw_batch_run_id") == batch_run_id)
    )

    df_source_count = df_source.count()
    df_target_count = df_target.count()
    assert df_source_count == df_target_count, (
        f"Row count mismatch: "
        f"{source_name}={df_source_count}, "
        f"{target_name}={df_target_count}"
    )

    print(f"✓ Row counts match ({df_source_count:,} records)")

    _validate_column_mappings(df_source,df_target,unified_json_file_path)

## execute registration testharness 